#### Feature engineering 
Feature engineering means creating new useful columns from the existing data so that we can answer business questions more easily.

In [1]:
import numpy as np
import pandas as pd

In [1]:
import pandas as pd

cleaned_data = pd.read_csv(
    "../data/processed/cleaned_transactions.csv"
)

cleaned_data["InvoiceDate"] = pd.to_datetime(
    cleaned_data["InvoiceDate"]
)

sales_data = cleaned_data[
    (cleaned_data["Quantity"] > 0) &
    (cleaned_data["Price"] > 0)
].copy()

print(sales_data.shape)

(1007914, 8)


In [2]:
# Create Revenue for each sales transaction
sales_data["Revenue"] = (
    sales_data["Quantity"] * sales_data["Price"]
)

# Check the result
sales_data[["Quantity", "Price", "Revenue"]].head()
# Check whether Revenue was calculated correctly
print(
    "Negative revenue:",
    (sales_data["Revenue"] < 0).sum()
)

print(
    "Zero revenue:",
    (sales_data["Revenue"] == 0).sum()
)

Negative revenue: 0
Zero revenue: 0


#### Important: 
- We are not supposed to depend on variables from another notebook. Each notebook should load the data it needs. 
- WHY?
- Each notebook has its own Python session, so variables like sales_data don't automatically carry over.
- Saving/loading the processed data makes each notebook independent and reproducible.

In [3]:
# Create date and time features
sales_data["Year"] = sales_data["InvoiceDate"].dt.year
sales_data["Month"] = sales_data["InvoiceDate"].dt.month
sales_data["Month_Name"] = sales_data["InvoiceDate"].dt.month_name()
sales_data["Day"] = sales_data["InvoiceDate"].dt.day
sales_data["Hour"] = sales_data["InvoiceDate"].dt.hour

In [4]:
sales_data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Year,Month,Month_Name,Day,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009,12,December,1,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,December,1,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,December,1,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009,12,December,1,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009,12,December,1,7


In [6]:
customer_data = sales_data.groupby("Customer ID").agg(
    Total_Revenue=("Revenue", "sum"),
    Total_Orders=("Invoice", "nunique"),
    Total_Quantity=("Quantity", "sum")
)
customer_data.head()

,Total_Revenue,Total_Orders,Total_Quantity
Customer ID,,,
12346.0,77556.46,12,74285
12347.0,4921.53,8,2967
12348.0,2019.40,5,2714
12349.0,4428.69,4,1624
12350.0,334.40,1,197


In [7]:
product_data = sales_data.groupby("StockCode").agg(
    Total_Revenue=("Revenue", "sum"),
    Total_Orders=("Invoice", "nunique"),
    Total_Quantity=("Quantity", "sum")
)
product_data.head()

,Total_Revenue,Total_Orders,Total_Quantity
StockCode,,,
10002,6942.26,362,8671
10002R,20.57,3,4
10080,129.29,27,315
10109,1.68,1,4
10120,142.74,73,664


In [8]:
# Final validation

print("Sales data shape:", sales_data.shape)
print("Duplicate rows:", sales_data.duplicated().sum())
print("Negative revenue:", (sales_data["Revenue"] < 0).sum())
print("Zero revenue:", (sales_data["Revenue"] == 0).sum())
print("Missing Revenue:", sales_data["Revenue"].isnull().sum())

print("\nCustomer data shape:", customer_data.shape)
print("Product data shape:", product_data.shape)


Sales data shape: (1007914, 14)
Duplicate rows: 0
Negative revenue: 0
Zero revenue: 0
Missing Revenue: 0

Customer data shape: (5878, 3)
Product data shape: (4917, 3)


In [9]:
sales_data.to_csv(
    "../data/processed/sales_features.csv",
    index=False
)
customer_data.to_csv(
    "../data/processed/customer_summary.csv",
    index=False
)

product_data.to_csv(
    "../data/processed/product_summary.csv",
    index=False
)